In [1]:
import polars as pl

giga = pl.read_csv(
    "../../data/expansion/raw/GigaPool_Sheet.xlsx - All Peptides.csv",
    null_values=["#N/A"],
)
exp = pl.read_parquet("../../data/expansion/triad/expansion_triad.parquet")

orig_tb_seq = (
    pl.concat(
        [
            pl.read_csv("../../data/expansion/raw/FLUproteintable.csv"),
            pl.read_csv(
                "../../data/expansion/raw/MTBproteintable.csv",
                new_columns=["ID", "sequence"],
            ),
        ]
    )
    .select("sequence")
    .unique()
)

## MHC formatting for MHCIIpan


In [53]:
m2n_ls = (
    exp.with_columns(
        (
            pl.when(pl.col("mhc_2_name").str.starts_with("DRB"))
            .then(pl.col("mhc_2_name").str.replace_all("*", "_", literal=True))
            .otherwise(
                pl.concat_str(
                    pl.lit("HLA-"), "mhc_1_name", pl.lit("-"), "mhc_2_name"
                ).str.replace_all("*", "", literal=True)
            )
            .str.replace_all(":", "")
        ).alias("tmp")
    )
    .select("tmp")
    .unique(maintain_order=True)
    .to_series()
    .to_list()
)


print(",".join([m2n for m2n in m2n_ls]))

HLA-DPA10103-DPB10201,HLA-DPA10301-DPB10402,HLA-DQA10301-DQB10302,HLA-DQA10102-DQB10602,DRB1_0403,DRB1_1503,DRB4_0103,DRB5_0101


## Write peptide FASTA


In [3]:
import math


## shorter than 15
short_pep = (
    exp.select("peptide")
    .filter(pl.col("peptide").str.len_chars() < 15)
    .select("peptide")
    .unique()
    .to_series()
    .to_list()
)
keep_pep = []
pep_context = []

for pep in short_pep:
    context = orig_tb_seq.filter(pl.col("sequence").str.contains(pep))
    if context.height == 0:
        print(f"No context found for {pep}")
        continue
    idx = context.select("sequence").to_series().str.find(pep).to_list()[0]
    keep_pep.append(pep)
    if idx == 0:
        pep_context.append(context.select("sequence").to_series()[0][:15])
    else:
        seq = context.select("sequence").to_series()[0]
        needed = 15 - len(pep)
        l = math.floor(needed / 2)
        r = math.ceil(needed / 2)
        pep_context.append(seq[idx - l : idx + len(pep) + r])

short_pep = pl.DataFrame({"peptide": keep_pep, "pep_with_context": pep_context})

# ## longer than 15
# exp.select("peptide").filter(pl.col("peptide").str.len_chars() > 15).select("peptide").unique().with_columns(
#     pl.col("peptide").map_elements(lambda x: )
# )

## concatenate
long_pep = (
    exp.select("peptide")
    .filter(pl.col("peptide").str.len_chars() >= 15)
    .select("peptide")
    .unique()
    .with_columns(pl.col("peptide").alias("pep_with_context"))
)

all_pep = pl.concat([short_pep, long_pep])

No context found for MSFVITAPELISA
No context found for PYKVKQNTLKLAT


In [88]:
with open("../../data/expansion/expansion_peptides.fasta", "w") as f:
    for row in all_pep.iter_rows(named=True):
        f.write(f">{row['peptide']}\n{row['pep_with_context']}\n")

## Get peptide lengths


In [93]:
l_list = (
    all_pep.select(pl.col("pep_with_context").str.len_chars())
    .unique()
    .sort(by="pep_with_context")
    .to_series()
    .to_list()
)

print(",".join([str(n) for n in l_list]))

15,16,17,18,19,20,22


## Load in NetMHCIIPan results


In [ ]:
m2n_ls = "DRB5_0101,HLA-DQA10301-DQB10302,HLA-DPA10301-DPB10402,DRB4_0103,HLA-DPA10103-DPB10201,HLA-DQA10102-DQB10602,DRB1_0403,DRB1_1503".split(
    ","
)

new_columns = ["Pos", "pep_with_context", "peptide", "target"]

for m2n in m2n_ls:
    new_columns.extend(
        [
            f"Core_{m2n}",
            f"Inverted_{m2n}",
            f"Score_{m2n}",
            f"Rank_{m2n}",
            f"Score_BA_{m2n}",
            f"nM_{m2n}",
            f"Rank_BA_{m2n}",
        ]
    )

new_columns += ["Ave", "NB"]

net = pl.read_csv(
    "../../data/expansion/2573277_NetMHCIIpan.xls",
    skip_rows=1,
    separator="\t",
    new_columns=new_columns,
)

# net = net.rename(
#     {
#         "Peptide": "pep_with_context",
#         "ID": "peptide",
#         # "Rank": "rank_el",
#         # "Score": "score_el"
#     }
# )

In [42]:
probe_el_pep = []
probe_el_m2n = []

for m2n in m2n_ls:
    if m2n.startswith("HLA"):
        m2n_fmt = m2n.split("-")[-1]
        m2n_fmt = m2n_fmt[:4] + "*" + m2n_fmt[4:6] + ":" + m2n_fmt[6:]
    else:
        m2n_fmt = m2n[:4] + "*" + m2n[5:7] + ":" + m2n[7:]

    mhc_info = net.select(
        [
            "Pos",
            "pep_with_context",
            "peptide",
            "target",
            f"Core_{m2n}",
            f"Inverted_{m2n}",
            f"Score_{m2n}",
            f"Rank_{m2n}",
            f"Score_BA_{m2n}",
            f"nM_{m2n}",
            f"Rank_BA_{m2n}",
        ]
    )

    binder = mhc_info.filter(pl.col(f"Rank_{m2n}") < 5)
    
    bpep = binder.select("peptide").unique().to_series().to_list()
    probe_el_pep.extend(bpep)
    probe_el_m2n.extend([m2n_fmt for _ in range(len(bpep))])

binding_interactions = pl.DataFrame(
    {"peptide": probe_el_pep, "mhc_2_name": probe_el_m2n}
)

In [ ]:
binding_interactions

peptide,mhc_2_name
str,str
"""GTTDNFQRYLQAASN""","""DRB5*01:01"""
"""NIRQAGVQYSRA""","""DRB5*01:01"""
"""EKGKIVKSVEMNAPN""","""DRB5*01:01"""
"""TYVLSIIPSGPLKAE""","""DRB5*01:01"""
"""QLIASHTAFAAKA""","""DPB1*02:01"""
…,…
"""ENRFIEIGVTRREVH""","""DRB4*01:03"""
"""FGQNTSAIAAAEAQY""","""DRB1*15:03"""
"""SGPLKAEIAQKLEDV""","""DPB1*02:01"""


In [ ]:
probes = pl.read_parquet("../../data/cresta/pmhc/staged/cresta_pmhc.remaining.parquet").select("peptide", "mhc_2_name").unique()

## TCR clustering

In [ ]:
from tcrtrifold.tcr import 